# Decorators

On Friday, `logged` called a function once, with whatever arguments arrived, and `make_adder` returned a function that kept a name from its enclosing scope. Put the two together, and we can make a **new** function that always logs.

**Predict:**

In [ ]:
def with_log(function):
    def wrapper(*args, **kwargs):
        print("calling", function.__name__, "with", args, kwargs)
        return function(*args, **kwargs)

    return wrapper


max_with_log = with_log(max)
print(max_with_log(3, 8, 5))

Python has a short form for this.

In [ ]:
@with_log
def add(a, b):
    "Add two Numbers"
    return a + b


print(add(1, 2))

`@with_log` above a `def` means: define the function, then `add = with_log(add)`. That's a **decorator**: a function that takes a function and returns one. You've seen `@pytest.mark.parametrize` already, and you'll meet `@dataclass` next week and `@app.get` with FastAPI.

**Predict:**

In [ ]:
print(add.__name__)

In [ ]:
help(add)

In [ ]:
import pytest


@pytest.mark.parametrize()
def test_nothing():
    "Docstring for my test"
    return True


print(test_nothing.__name__)

In [ ]:
help(test_nothing)

The name `add` now refers to `wrapper`, so that's what `__name__` and `help` report. The standard library has a decorator that copies the name and docstring across, and it's worth using whenever you write a decorator:

In [ ]:
from functools import wraps


def with_log(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print("calling", function.__name__, "with", args, kwargs)
        return function(*args, **kwargs)

    return wrapper


@with_log
def add(a, b):
    return a + b


print(add.__name__)

## A decorated function that calls itself

`countdown` from Friday, with `@with_log` on top. **Predict:**

In [ ]:
@with_log
def countdown(n):
    if n == 0:
        print("go!")
    else:
        print(n)
        countdown(n - 1)


countdown(2)

Inside `countdown`, the name `countdown` is looked up when the call happens, not when the function is defined (Friday: "when Python looks"). By then the global name `countdown` refers to the wrapper, so every recursive call goes through the wrapper too.

## Timing

A decorator can do something before **and** after the call. **Predict:** which line comes first, the time or the sum?

In [ ]:
import time


def timed(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = function(*args, **kwargs)
        print(
            function.__name__,
            args,
            "took",
            round(time.perf_counter() - start, 3),
            "seconds",
        )
        return result

    return wrapper


@timed
def slow_sum(n):
    total = 0
    for i in range(n):
        total += i
    return total


print(slow_sum(10_000_000))

`print(slow_sum(...))` has to call `slow_sum` before it can print anything, and the wrapper prints the time before it returns. The wrapper must `return result`: without that line, every decorated function would return `None`.

**Predict:** what does this print, and in which order?

In [ ]:
@timed
def countdown(n):
    if n == 0:
        print("go!")
    else:
        print(n)
        countdown(n - 1)


countdown(2)

Every level is timed, and the times come out innermost first, `(0,)` before `(2,)`: the wrapper prints **after** its call has finished, like `mystery` on Friday. The outermost time includes all the others.

## Counting calls

On Friday, `ways_counted` counted its own calls: the counting line sat inside the function. A decorator can count for any function without changing it. `calls` is a dict outside, as on Friday, with one entry per function name.

In [ ]:
calls = {}


def count_calls(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        calls[function.__name__] = calls.get(function.__name__, 0) + 1
        return function(*args, **kwargs)

    return wrapper


@count_calls
def ways(n):
    if n <= 1:
        return 1
    return ways(n - 1) + ways(n - 2)


print(ways(10))

**Predict:**

In [ ]:
print(calls)

Every recursive call goes through the wrapper, as with `countdown`, so the count is the 177 from Friday's table.

**Predict:**

In [ ]:
ways(10)
print(calls)

354: the dict isn't reset between runs, so the second run adds to the first. Friday's loop set `calls["count"] = 0` before each run for this reason. A counter that's never reset is a classic way to fool yourself while measuring.

## Watching the calls

**Predict:** how many lines does `ways(4)` log?

In [ ]:
@with_log
def ways(n):
    if n <= 1:
        return 1
    return ways(n - 1) + ways(n - 2)


ways(4)

Nine, the tree from Friday's board: `ways(3)` once, `ways(2)` twice, `ways(1)` three times, `ways(0)` twice. Putting a log or a counter on a function, without changing the function itself, is a quick way to see what your own code really does.

## A decorator with brackets

`@pytest.mark.parametrize(...)` has brackets after the name. **Predict:**

In [ ]:
def repeat(times):
    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = function(*args, **kwargs)
            return result

        return wrapper

    return decorator


@repeat(3)
def cheer(name):
    print("go", name)


cheer("Ada")

`@repeat(3)` is called first, and what it returns is the decorator: `cheer = repeat(3)(cheer)`. Three levels of `def` is about as deep as decorators go. You'll use them much more often than you write them: `@pytest.mark.parametrize` (you've used it already), `@dataclass` on Friday, `@app.get` with FastAPI later.